# 03. AutoIntent Few-Shot for Jailbreak Detection

Эксперименты AutoIntent в few-shot режиме на WildJailbreak.

## Содержание
1. Setup
2. Few-shot runs (n_shots × seeds)
3. Результаты — таблица из metrics.json
4. Scaling curve — F1 по n_shots
5. Variance analysis — std по seeds

## 1. Setup

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install -q autointent python-dotenv

In [ ]:
# Load environment variables from .env (macOS ARM fix)
from dotenv import load_dotenv
load_dotenv("../../../.env")  # project root

import sys
import os
import subprocess
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project paths
NOTEBOOK_DIR = Path.cwd()
TASK_DIR = NOTEBOOK_DIR.parent
PROJECT_ROOT = TASK_DIR.parent.parent
SCRIPTS_DIR = TASK_DIR / "scripts"
PROCESSED = TASK_DIR / "data" / "processed"
RESULTS = TASK_DIR / "results"
RUNS = TASK_DIR / "runs"

sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir: {RESULTS}")
print(f"Runs dir: {RUNS}")
print(f"\nOMP_NUM_THREADS={os.environ.get('OMP_NUM_THREADS', 'not set')}")

## 2. Few-shot runs

In [ ]:
# Configuration
N_SHOTS_LIST = [10, 20, 50]
SEEDS = [42, 123, 456]

# Set to True to force retraining even if model exists
FORCE_RERUN = False

# Set to True to use small embedder (faster but not comparable)
PILOT_MODE = False

In [ ]:
def get_model_dir(n_shots: int, seed: int, pilot: bool = False) -> Path:
    """Get model directory path."""
    model_name = "autointent_classic-light_pilot" if pilot else "autointent_classic-light"
    return RUNS / f"{model_name}_{n_shots}shot_seed{seed}"


def check_run_exists(n_shots: int, seed: int, pilot: bool = False) -> bool:
    """Check if run already completed (model dir exists with metadata)."""
    model_dir = get_model_dir(n_shots, seed, pilot)
    metadata_path = model_dir / "train_metadata.json"
    return metadata_path.exists()


def run_autointent(n_shots: int, seed: int, pilot: bool = False) -> int:
    """Run AutoIntent training and evaluation."""
    cmd = [
        sys.executable,
        str(SCRIPTS_DIR / "run_autointent.py"),
        "--mode", "fewshot",
        "--n_shots", str(n_shots),
        "--seed", str(seed),
    ]
    if pilot:
        cmd.append("--pilot")
    
    result = subprocess.run(cmd, cwd=str(TASK_DIR))
    return result.returncode

In [ ]:
# Run all combinations
total_runs = len(N_SHOTS_LIST) * len(SEEDS)
completed = 0
skipped = 0
failed = 0

print(f"Total runs planned: {total_runs}")
print(f"FORCE_RERUN: {FORCE_RERUN}")
print(f"PILOT_MODE: {PILOT_MODE}")
print("=" * 60)

for n_shots in N_SHOTS_LIST:
    for seed in SEEDS:
        run_id = f"{n_shots}shot_seed{seed}"
        
        # Check if already exists
        if not FORCE_RERUN and check_run_exists(n_shots, seed, PILOT_MODE):
            print(f"[SKIP] {run_id} — already exists")
            skipped += 1
            continue
        
        print(f"\n[RUN] {run_id}")
        print("-" * 40)
        
        returncode = run_autointent(n_shots, seed, PILOT_MODE)
        
        if returncode == 0:
            print(f"[OK] {run_id} completed")
            completed += 1
        else:
            print(f"[FAIL] {run_id} failed with code {returncode}")
            failed += 1

print("\n" + "=" * 60)
print(f"Summary: completed={completed}, skipped={skipped}, failed={failed}")

## 3. Результаты — таблица из metrics.json

In [ ]:
# Load results
metrics_path = RESULTS / "metrics.json"

if metrics_path.exists():
    with open(metrics_path, "r") as f:
        all_results = json.load(f)
    
    # Handle both list and dict formats
    if isinstance(all_results, dict):
        all_results = list(all_results.values())
    
    # Filter AutoIntent results
    ai_results = [r for r in all_results if "autointent" in r.get("model_name", "").lower()]
    
    if ai_results:
        df = pd.DataFrame(ai_results)
        
        # Select and order columns
        cols = ["n_shots", "seed", "f1", "precision", "recall", 
                "over_refusal_rate", "recall_adversarial_harmful"]
        cols = [c for c in cols if c in df.columns]
        
        df_display = df[cols].sort_values(["n_shots", "seed"])
        
        print(f"AutoIntent results: {len(df_display)} runs")
        display(df_display)
    else:
        print("No AutoIntent results found in metrics.json")
else:
    print(f"No results file found at {metrics_path}")
    print("Run the training first (section 2).")

## 4. Scaling Curve — F1 по n_shots

In [ ]:
# Aggregate by n_shots (mean ± std across seeds)
if metrics_path.exists() and 'df' in dir() and len(df) > 0:
    scaling_data = []
    
    for n_shots in N_SHOTS_LIST:
        subset = df[df["n_shots"] == n_shots]
        if len(subset) > 0:
            scaling_data.append({
                "n_shots": n_shots,
                "f1_mean": subset["f1"].mean(),
                "f1_std": subset["f1"].std() if len(subset) > 1 else 0,
                "recall_mean": subset["recall"].mean(),
                "recall_std": subset["recall"].std() if len(subset) > 1 else 0,
                "n_seeds": len(subset),
            })
    
    scaling_df = pd.DataFrame(scaling_data)
    
    print("AutoIntent: F1 и Recall по n_shots (mean ± std)")
    print("=" * 60)
    for _, row in scaling_df.iterrows():
        f1_str = f"{row['f1_mean']:.3f} ± {row['f1_std']:.3f}" if row['f1_std'] > 0 else f"{row['f1_mean']:.3f}"
        recall_str = f"{row['recall_mean']:.3f} ± {row['recall_std']:.3f}" if row['recall_std'] > 0 else f"{row['recall_mean']:.3f}"
        print(f"  {int(row['n_shots']):>2}-shot: F1 = {f1_str}, Recall = {recall_str} ({int(row['n_seeds'])} seeds)")
else:
    print("No data available for scaling curve")

In [ ]:
# Plot scaling curve
if 'scaling_df' in dir() and len(scaling_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # F1 plot
    ax = axes[0]
    x = scaling_df["n_shots"]
    ax.errorbar(x, scaling_df["f1_mean"], yerr=scaling_df["f1_std"], 
                fmt='o-', capsize=5, capthick=2, markersize=8, linewidth=2,
                color='#2ecc71', label='AutoIntent')
    
    # Add value labels
    for i, row in scaling_df.iterrows():
        ax.annotate(f"{row['f1_mean']:.3f}", 
                    (row['n_shots'], row['f1_mean']),
                    textcoords="offset points", xytext=(0, 10),
                    ha='center', fontsize=10)
    
    ax.set_xlabel("n_shots (per class)", fontsize=12)
    ax.set_ylabel("F1 (jailbreak class)", fontsize=12)
    ax.set_title("F1 Scaling Curve", fontsize=14)
    ax.set_xticks(N_SHOTS_LIST)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    # Recall plot
    ax = axes[1]
    ax.errorbar(x, scaling_df["recall_mean"], yerr=scaling_df["recall_std"],
                fmt='s-', capsize=5, capthick=2, markersize=8, linewidth=2,
                color='#3498db', label='AutoIntent')
    
    for i, row in scaling_df.iterrows():
        ax.annotate(f"{row['recall_mean']:.3f}",
                    (row['n_shots'], row['recall_mean']),
                    textcoords="offset points", xytext=(0, 10),
                    ha='center', fontsize=10)
    
    ax.set_xlabel("n_shots (per class)", fontsize=12)
    ax.set_ylabel("Recall (jailbreak class)", fontsize=12)
    ax.set_title("Recall Scaling Curve", fontsize=14)
    ax.set_xticks(N_SHOTS_LIST)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(RESULTS / "scaling_curve_fewshot.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved to {RESULTS / 'scaling_curve_fewshot.png'}")
else:
    print("No data available for plotting")

## 5. Variance Analysis — std по seeds

In [ ]:
# Variance analysis
if 'scaling_df' in dir() and len(scaling_df) > 0:
    print("Variance Analysis (CV = std/mean)")
    print("=" * 60)
    
    for _, row in scaling_df.iterrows():
        cv_f1 = row['f1_std'] / row['f1_mean'] * 100 if row['f1_mean'] > 0 else 0
        cv_recall = row['recall_std'] / row['recall_mean'] * 100 if row['recall_mean'] > 0 else 0
        
        print(f"\n{int(row['n_shots'])}-shot ({int(row['n_seeds'])} seeds):")
        print(f"  F1:     {row['f1_mean']:.3f} ± {row['f1_std']:.3f}  (CV = {cv_f1:.1f}%)")
        print(f"  Recall: {row['recall_mean']:.3f} ± {row['recall_std']:.3f}  (CV = {cv_recall:.1f}%)")
    
    # Overall stability
    avg_cv = scaling_df['f1_std'].mean() / scaling_df['f1_mean'].mean() * 100
    print(f"\nOverall F1 CV: {avg_cv:.1f}%")
    
    if avg_cv < 5:
        print("Verdict: STABLE (CV < 5%)")
    elif avg_cv < 10:
        print("Verdict: MODERATE (CV 5-10%)")
    else:
        print("Verdict: UNSTABLE (CV > 10%)")
else:
    print("No data available for variance analysis")

In [ ]:
# Detailed per-seed results
if 'df' in dir() and len(df) > 0:
    print("\nDetailed results per seed:")
    print("=" * 60)
    
    pivot = df.pivot_table(
        index='n_shots', 
        columns='seed', 
        values='f1',
        aggfunc='first'
    )
    
    if not pivot.empty:
        # Add mean and std columns
        pivot['mean'] = pivot.mean(axis=1)
        pivot['std'] = pivot.std(axis=1)
        
        display(pivot.round(3))